<p align="center"><img src="https://www.sap.com/dam/application/shared/logos/sap-logo-svg.svg" width="110" /></p>

# SAP TechEd 2026 — TabPFN-3.5 Plus + LangChain Agent (Google Colab)

**Based on:** [SAP Developer Center tutorial](https://developers.sap.com/tutorials/ai-core-prior-labs-langgraph) (16 steps). Adapted for **Google Colab**, no VS Code required.

**Goal:** Use SAP AI Core to predict `DELAY_DAYS` (regression) and `LATE` (classification), generate an LLM logistics report, and evaluate MAE / accuracy.

**Important:** SAP TechEd temporary service keys may expire. Never share your client secret, token, or downloaded service-key ZIP in a public notebook. This notebook does not save secrets into the `.ipynb` file.

**Run cells top-to-bottom.** Requires working SAP AI Core access and the TabPFN deployment.

**This edition supports the uploaded encrypted SAP TechEd service-key ZIP.** Keep the ZIP and password private. Do not save an executed notebook containing credential output.

## 1 — Install dependencies (SAP Step 7)
Colab already provides an isolated Python runtime, so no `venv` or VS Code is needed.

In [ ]:
%pip -q install langchain langchain-litellm requests pandas python-dotenv mcp

## 2 — Load `payload_data.py` (SAP Steps 3 and 10)

Upload the `payload_data.py` file you already downloaded, using Colab's **Files** pane, or let the notebook download the official file automatically. The payload has 50 known training rows and 10 target-specific test rows.

In [ ]:
from pathlib import Path
import urllib.request, importlib.util

DATA_URL = "https://raw.githubusercontent.com/sap-tutorials/Tutorials/master/tutorials/ai-core-prior-labs-langgraph/payload_data.py"
if not Path("payload_data.py").exists():
    try:
        urllib.request.urlretrieve(DATA_URL, "payload_data.py")
        print("Downloaded official payload_data.py")
    except Exception as exc:
        print("Automatic download failed:", exc)
        print("Upload payload_data.py using the Colab Files pane, then rerun this cell.")
        # from google.colab import files
        # files.upload()

assert Path("payload_data.py").exists(), "Upload payload_data.py and rerun."
from payload_data import payload, GROUND_TRUTH
import pandas as pd
print("Rows:", len(payload["rows"]))
print("Targets:", [t["name"] for t in payload["prediction_config"]["target_columns"]])
display(pd.DataFrame(payload["rows"]).head())

## 3 — SAP AI Core credentials (SAP Steps 1, 4, 6)

**Choose one of the two methods below:**

- **Method A (recommended for your TechEd ZIP):** Upload only `sk-20261028144122.zip` to the temporary Colab session (Files pane or upload prompt). Run the next cell and type the ZIP password into a hidden prompt. Credentials remain in runtime memory and are **not written into the notebook**.
- **Method B:** Use Colab Secrets for `AICORE_AUTH_URL`, `AICORE_CLIENT_ID`, `AICORE_CLIENT_SECRET`, `AICORE_BASE_URL`. Enable Notebook access for each.

**Do not paste your password, token, ZIP, or JSON service key into chat or a shared notebook.** ZIPs rotate frequently; a newer key may be needed. The shared deployment ID from the SAP tutorial is `d845a8b8465ba1e6`.

**Security:** This is a short-lived Colab runtime. Avoid sharing the notebook's runtime, and clear its session when finished.

In [ ]:
import os, json, zipfile, io, getpass
from pathlib import Path

# Select credential loading method: "zip" or "secrets"
CREDENTIAL_METHOD = "zip"
os.environ.setdefault("AICORE_RESOURCE_GROUP", "default")
os.environ.setdefault("TABPFN_DEPLOYMENT_ID", "d845a8b8465ba1e6")
os.environ.setdefault("LLM_MODEL", "sap/amazon--nova-lite")

if CREDENTIAL_METHOD == "zip":
    from google.colab import files
    zip_files = sorted(Path('/content').glob('sk-*.zip'))
    if not zip_files:
        print('Upload your newest TechEd service-key ZIP now (do not upload it publicly).')
        uploaded = files.upload()
        zip_files = sorted(Path('/content').glob('sk-*.zip'))
    if not zip_files:
        raise FileNotFoundError('No sk-*.zip found. Upload the service-key ZIP.')
    zip_path = zip_files[-1]  # Timestamped filenames sort chronologically.
    print('Selected:', zip_path.name)
    zip_password = getpass.getpass('Enter ZIP password (hidden): ')
    try:
        with zipfile.ZipFile(zip_path) as archive:
            json_files = [name for name in archive.namelist() if name.lower().endswith('.json')]
            if len(json_files) != 1:
                raise ValueError('Expected exactly one JSON service key in ZIP.')
            service_key = json.loads(archive.read(json_files[0], pwd=zip_password.encode()))
    finally:
        del zip_password

    # Standard SAP AI Core service-key shape: clientid/clientsecret/url/serviceurls.AI_API_URL.
    # Also handles an OAuth "uaa" nested object, if present.
    auth = service_key.get('uaa', service_key)
    client_id = auth.get('clientid') or auth.get('client_id')
    client_secret = auth.get('clientsecret') or auth.get('client_secret')
    oauth_base = auth.get('url') or auth.get('auth_url')
    serviceurls = service_key.get('serviceurls', {})
    api_base = (serviceurls.get('AI_API_URL') or service_key.get('AI_API_URL')
                or service_key.get('apiurl') or service_key.get('api_url'))
    if not all([client_id, client_secret, oauth_base, api_base]):
        raise ValueError('Unrecognized service-key JSON structure. Ask for help with FIELD NAMES ONLY; never share values.')
    os.environ['AICORE_CLIENT_ID'] = client_id
    os.environ['AICORE_CLIENT_SECRET'] = client_secret
    os.environ['AICORE_AUTH_URL'] = oauth_base.rstrip('/') + ('' if oauth_base.rstrip('/').endswith('/oauth/token') else '/oauth/token')
    os.environ['AICORE_BASE_URL'] = api_base.rstrip('/')
    del service_key, auth, client_secret, client_id
elif CREDENTIAL_METHOD == "secrets":
    from google.colab import userdata
    for key in ['AICORE_AUTH_URL', 'AICORE_CLIENT_ID', 'AICORE_CLIENT_SECRET', 'AICORE_BASE_URL']:
        value = userdata.get(key)
        if not value:
            raise ValueError('Missing Colab Secret: ' + key)
        os.environ[key] = value.strip()
else:
    raise ValueError('CREDENTIAL_METHOD must be zip or secrets')

assert os.environ['AICORE_AUTH_URL'].startswith('https://')
assert os.environ['AICORE_BASE_URL'].startswith('https://')
TABPFN_DEPLOYMENT_ID = os.environ['TABPFN_DEPLOYMENT_ID']
print('Credentials loaded successfully (values hidden).')
print('Deployment ID:', TABPFN_DEPLOYMENT_ID)


## 4 — OAuth token test (SAP Step 9)
Checks that the credentials work. **Never print the access token.**

In [ ]:
import requests

def _get_auth_token() -> str:
    response = requests.post(
        os.environ["AICORE_AUTH_URL"],
        data={
            "grant_type": "client_credentials",
            "client_id": os.environ["AICORE_CLIENT_ID"],
            "client_secret": os.environ["AICORE_CLIENT_SECRET"],
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["access_token"]

try:
    _ = _get_auth_token()
    print("OAuth authentication succeeded. Token is hidden.")
except requests.RequestException as exc:
    raise RuntimeError("SAP authentication failed. Check Secrets, token URL, key expiry, and notebook access.") from exc

## 5 — Prepare features (SAP Step 11)
Both targets are excluded from features to avoid target leakage.

In [ ]:
import json
FEATURE_COLS = [
    "CATEGORY", "DISTANCE_KM", "WEIGHT_KG", "ORDER_VALUE", "PROMISED_DAYS",
    "WAREHOUSE_HRS", "SHIP_METHOD", "CARRIER", "PKG_SIZE", "PRIORITY", "WEATHER", "SHIP_COST",
]
TABPFN_DEPLOYMENT_ID = os.environ["TABPFN_DEPLOYMENT_ID"]
print("Feature count:", len(FEATURE_COLS))
for target in payload["prediction_config"]["target_columns"]:
    c, p = target["name"], target["prediction_placeholder"]
    print(c, "training:", sum(r.get(c) is not None and r.get(c) != p for r in payload["rows"]),
          "test:", sum(r.get(c) == p for r in payload["rows"]))

## 6 — TabPFN REST prediction function (SAP Step 11)
This follows the official tutorial's per-target request structure: numeric `y_train` for regression and named-label dictionary for classification. A failed API request raises an error rather than generating fabricated predictions.

In [ ]:
def call_tabpfn(payload: dict) -> str:
    token = _get_auth_token()
    url = (f"{os.environ['AICORE_BASE_URL'].rstrip('/')}/v2/inference/deployments/"
           f"{TABPFN_DEPLOYMENT_ID}/predict")
    headers = {
        "Authorization": f"Bearer {token}",
        "AI-Resource-Group": os.environ.get("AICORE_RESOURCE_GROUP", "default"),
        "Content-Type": "application/json",
    }
    rows = payload["rows"]
    results = {"predictions": {}}
    for target in payload["prediction_config"]["target_columns"]:
        col_name = target["name"]
        placeholder = target["prediction_placeholder"]
        task_type = target["task_type"]
        train_rows = [r for r in rows if r.get(col_name) is not None and r.get(col_name) != placeholder]
        test_rows = [r for r in rows if r.get(col_name) == placeholder]
        if not test_rows:
            results["predictions"][col_name] = []
            continue
        train_df, test_df = pd.DataFrame(train_rows), pd.DataFrame(test_rows)
        x_train = {col: train_df[col].tolist() for col in FEATURE_COLS}
        x_test = {col: test_df[col].tolist() for col in FEATURE_COLS}
        if task_type == "regression":
            request_payload = {
                "task_config": {"task": "regression", "predict_params": {"output_type": "mean"}},
                "x_train": x_train,
                "y_train": train_df[col_name].astype(float).tolist(),
                "x_test": x_test,
            }
        else:
            request_payload = {
                "task_config": {
                    "task": "classification",
                    "tabpfn_config": {"n_estimators": 8, "inference_precision": "auto"},
                    "predict_params": {"output_type": "preds"},
                },
                "x_train": x_train,
                "y_train": {col_name: train_df[col_name].tolist()},
                "x_test": x_test,
            }
        print(f"Calling TabPFN [{col_name}]...")
        response = requests.post(url, headers=headers, json=request_payload, timeout=300)
        print("HTTP status:", response.status_code)
        response.raise_for_status()
        predictions = response.json()["prediction"]
        if len(predictions) != len(test_rows):
            raise ValueError(f"Expected {len(test_rows)} predictions for {col_name}, got {len(predictions)}")
        results["predictions"][col_name] = [
            {"ORD_ID": row["ORD_ID"], "SHIP_METHOD": row["SHIP_METHOD"], col_name: p}
            for row, p in zip(test_rows, predictions)
        ]
    return json.dumps(results, indent=2)

## 7 — Run TabPFN directly (recommended diagnostic)
Verify both predictions before involving the LLM. These are real SAP API calls, not a local model. The tutorial estimates 10–60 seconds per target.

In [ ]:
raw_predictions = call_tabpfn(payload)
print(raw_predictions[:6000])

## 8 — Evaluate against ground truth (SAP Step 13)

In [ ]:
def evaluate_predictions(predictions_json: str) -> dict:
    preds = json.loads(predictions_json)["predictions"]
    metrics = {}
    if "DELAY_DAYS" in preds:
        errors = []
        for item in preds["DELAY_DAYS"]:
            truth = GROUND_TRUTH.get(item["ORD_ID"], {}).get("DELAY_DAYS")
            if truth is not None:
                errors.append(abs(float(item["DELAY_DAYS"]) - float(truth)))
        if errors:
            metrics["MAE_days"] = sum(errors) / len(errors)
            print(f"MAE (DELAY_DAYS): {metrics['MAE_days']:.2f} days")
    if "LATE" in preds:
        matches = []
        for item in preds["LATE"]:
            truth = GROUND_TRUTH.get(item["ORD_ID"], {}).get("LATE")
            if truth is not None:
                matches.append(str(item["LATE"]).strip().lower() == str(truth).strip().lower())
        if matches:
            metrics["Accuracy_LATE"] = sum(matches) / len(matches)
            print(f"Accuracy (LATE): {sum(matches)}/{len(matches)} = {metrics['Accuracy_LATE']:.0%}")
    return metrics

metrics = evaluate_predictions(raw_predictions)

## 9 — LangChain agent (SAP Steps 8 and 12)

This is the SAP tutorial's agent path. **The model name `sap/amazon--nova-lite` may require additional SAP-specific LiteLLM routing/credentials.** If it is not configured, the next cell will explain the failure. Your TabPFN predictions and evaluation from earlier cells are still valid; use the deterministic report fallback below.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_litellm import ChatLiteLLM

@tool
def run_tabpfn_predictions() -> str:
    """Run TabPFN-3.5 Plus predictions for shipment delay days (DELAY_DAYS)
    and on-time status (LATE). Returns JSON predictions for all test rows."""
    return call_tabpfn(payload)

system_prompt = """You are a senior e-commerce logistics analyst.
You have received TabPFN-3.5 Plus predictions for two missing fields in a shipment dataset:
- DELAY_DAYS: predicted number of days after promised delivery.
- LATE (Yes/No): whether each order arrives after the promised date.
Summarise the predictions in a clear operations report:
- List delivery delay for each order, grouped by shipping method.
- List predicted on-time/late status for each order.
- Highlight patterns in shipping methods, carriers or weather only when supported by the data.
Rely exclusively on model predictions. Never invent or adjust predicted values."""

try:
    model = ChatLiteLLM(model=os.environ.get("LLM_MODEL", "sap/amazon--nova-lite"), temperature=0)
    forecast_agent = create_agent(model=model, tools=[run_tabpfn_predictions], system_prompt=system_prompt)
    print('LangChain agent initialized. Actual LLM connectivity is tested on invocation.')
except Exception as exc:
    forecast_agent = None
    print('Agent initialization unavailable:', type(exc).__name__, str(exc)[:350])

## 10 — Run agent and print logistics report (SAP Steps 14–15)
This triggers fresh SAP API predictions via the agent tool, then asks the LLM to write the report. If it fails due to LLM access, use the direct predictions from Step 7; do not mistake a missing report for a TabPFN failure.

In [ ]:
agent_result = None
if forecast_agent is None:
    print('LLM agent was not initialized. Continue with the fallback report below.')
else:
    try:
        agent_result = forecast_agent.invoke({"messages": [{"role": "user", "content":
            "Run the TabPFN predictions and write a concise logistics operations summary."}]})
        print('\nE-commerce Logistics Report (LLM):')
        print(agent_result['messages'][-1].content)
    except Exception as exc:
        print('LLM call unavailable:', type(exc).__name__, str(exc)[:500])
        print('TabPFN results remain available; continue to deterministic report.')


## 11 — Optional: Inspect results in a table and export (Colab addition)
Use `raw_predictions` from the direct API call to inspect individual orders, even if the LLM report is unavailable.

In [ ]:
pred_dict = json.loads(raw_predictions)["predictions"]
for target, items in pred_dict.items():
    print("\n", target)
    display(pd.DataFrame(items))

# Optional CSV exports (no secrets)
for target, items in pred_dict.items():
    pd.DataFrame(items).to_csv(f"tabpfn_{target.lower()}_predictions.csv", index=False)
print("Prediction CSVs saved in Colab's temporary working directory.")

## Troubleshooting / resources

- **Missing Secrets:** Use Colab's left-side key icon, add all four required names, and enable notebook access.
- **401 / 403:** Check service key expiry, client credentials, token URL, and resource-group permissions.
- **404:** Verify API base URL, deployment ID and whether the shared deployment is still running.
- **LLM / LiteLLM error:** The event's `sap/amazon--nova-lite` model may require a separately configured SAP-compatible LiteLLM endpoint. Ask event staff for current settings. The direct TabPFN predictions and evaluation can still be used.
- **Data file:** [payload_data.py](https://github.com/sap-tutorials/Tutorials/blob/master/tutorials/ai-core-prior-labs-langgraph/payload_data.py)
- **Original agent:** [agent-prior-labs.py](https://github.com/sap-tutorials/Tutorials/blob/master/tutorials/ai-core-prior-labs-langgraph/agent-prior-labs.py)
- **Official tutorial:** [SAP Developer Center](https://developers.sap.com/tutorials/ai-core-prior-labs-langgraph)
- **Temporary TechEd keys:** [SAP-samples/teched2026-sk-gen](https://github.com/SAP-samples/teched2026-sk-gen/tree/main/keys)

**Security:** Never upload `.env`, service-key JSON, ZIP passwords, access tokens, or credential-bearing notebook outputs to public repositories. Colab sessions are temporary; downloaded files and generated CSVs disappear when the runtime resets unless you save them.

## 12 — Guaranteed report from actual TabPFN results (no LLM needed)

This is a deterministic summary, **not an LLM-generated report**. It uses the real predictions returned by SAP AI Core in section 7 and avoids fabricated predictions.

In [ ]:
import pandas as pd
prediction_obj = json.loads(raw_predictions)['predictions']
for target in ('DELAY_DAYS', 'LATE'):
    frame = pd.DataFrame(prediction_obj.get(target, []))
    print(f'\n=== {target} predictions ===')
    if frame.empty:
        print('No predictions returned.')
    else:
        if 'SHIP_METHOD' in frame:
            frame = frame.sort_values('SHIP_METHOD')
        display(frame.reset_index(drop=True))
print('\n=== Evaluation ===')
metrics = evaluate_predictions(raw_predictions)
print(metrics)


## 13 — Save non-secret outputs

Save predictions and metrics only. **Do not export the decrypted service key, OAuth token, or ZIP password.**

In [ ]:
from pathlib import Path
Path('sap_tabpfn_predictions.json').write_text(raw_predictions, encoding='utf-8')
Path('sap_tabpfn_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved sap_tabpfn_predictions.json and sap_tabpfn_metrics.json')
# Optional download:
# from google.colab import files
# files.download('sap_tabpfn_predictions.json')
